In [1]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd

WORK_PATH = Path("/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/soybean_gene_region/")

In [4]:
# 提取gff3基因

with open(WORK_PATH / "glyma.Wm82.gnm1.ann1.DvBy.gene_models_main.gff3", "r", encoding="utf-8") as fin, \
     open(WORK_PATH / "all_gene_list.txt", "w", encoding="utf-8") as fout:

    fout.write("Gene\tChr\tStart\tEnd\n")

    for line in fin:
        # 跳过注释行和空行
        if line.startswith("#") or not line.strip():
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) < 9:
            continue

        seqid = fields[0]
        feature_type = fields[2]
        start = fields[3]
        end = fields[4]
        attributes = fields[8]

        # 只提取 gene
        if feature_type != "gene":
            continue

        # -------------------------
        # 只保留 Gm01-Gm20
        # -------------------------
        chr_part = seqid.split(".")[-1]

        match_chr = re.fullmatch(r"Gm(\d{2})", chr_part)

        if not match_chr:
            continue

        chr_name = int(match_chr.group(1))

        # 再保险，只保留 1-20
        if not 1 <= chr_name <= 20:
            continue

        # 提取基因名称，例如 Glyma01g00210
        match = re.search(r"(?:^|;)Name=([^;]+)", attributes)

        if match:
            gene_name = match.group(1)

            # glyma.Wm82.gnm1.Gm01 → Gm01 → 1
            chr_name = int(
                seqid.split(".")[-1].replace("Gm", "")
            )

            fout.write(
                f"{gene_name}\t{chr_name}\t{start}\t{end}\n"
            )

print(f"完成，结果保存到: all_gene_list.txt")

完成，结果保存到: all_gene_list.txt


In [2]:
# =========================
# 2. 读取 Supplementary Table S1
# =========================
# 注意：这个Excel的真正表头位于第2行，所以使用 header=1
df = pd.read_excel(WORK_PATH / "pbi13682-sup-0001-tables1.xlsx",sheet_name="Sheet1",header=1)

# =========================
# 3. 提取 Accession number
# =========================
gene_ids = []

for value in df["Accession number"].dropna():

    value = str(value).strip()

    if not value:
        continue

    # 处理：
    # Glyma02g43700 OR Glyma02g43680
    #
    # 大小写均可识别 OR / or / Or
    ids = re.split(r"\s+OR\s+", value, flags=re.IGNORECASE)

    for gene_id in ids:

        gene_id = gene_id.strip()

        if gene_id:
            gene_ids.append(gene_id)


# =========================
# 4. 去重，同时保持原来的顺序
# =========================
gene_ids = list(dict.fromkeys(gene_ids))

print(f"\n提取到的唯一基因ID数量: {len(gene_ids)}")


# =========================
# 5. 读取 all_gene_list.txt
# =========================
# 文件格式预计是：
#
# Gene    Chr    Start    End
# Glyma01g00210    1    27643    27977
# ...

gene_position_df = pd.read_csv(WORK_PATH / "all_gene_list.txt",sep="\t")

print("\nall_gene_list.txt columns:")
print(gene_position_df.columns.tolist())


# =========================
# 6. 和功能基因ID进行匹配
# =========================
result_df = gene_position_df[
    gene_position_df["Gene"].isin(gene_ids)
].copy()

result_df.to_csv(WORK_PATH.parent / "soybean-region.txt",
    sep="\t",
    index=False
)


提取到的唯一基因ID数量: 227

all_gene_list.txt columns:
['Gene', 'Chr', 'Start', 'End']


In [6]:
result_df[result_df["End"] <= result_df["Start"]]

,Gene,Chr,Start,End


In [4]:
result_df

,Gene,Chr,Start,End
85,Glyma01g01190,1,837438,839839
569,Glyma01g07120,1,7694028,7700472
595,Glyma01g07580,1,8387062,8389448
1475,Glyma01g35910,1,48361828,48364787
1583,Glyma01g37060,1,49439172,49440019
...,...,...,...,...
52574,Glyma20g24590,20,34219908,34221556
52615,Glyma20g25000,20,34688253,34690758
53403,Glyma20g33850,20,42326422,42327389
53529,Glyma20g35180,20,43456228,43459054


In [5]:
# Old
# File 1
"""
df_1 = pd.read_excel(WORK_PATH / "1-soybean-yield-region.xlsx")

region_df_1 = df_1[["Chr.", "Start position", "End position"]].copy()

# 去除 Start position == End position 的行
region_df_1 = region_df_1[region_df_1["Start position"] != region_df_1["End position"]].copy()

# 重命名列
region_df_1 = region_df_1.rename(columns={"Chr.": "Chr","Start position": "Start", "End position": "End",})
"""
df_1 = pd.read_excel(WORK_PATH / "1-soybean-yield-region-1.xlsx")

region_df_1 = df_1[["Chr", "Start (bp)", "Stop (bp)"]].copy()

# 去除 Start position == End position 的行
region_df_1 = region_df_1[region_df_1["Start (bp)"] != region_df_1["Stop (bp)"]].copy()

# 重命名列
region_df_1 = region_df_1.rename(columns={"Start (bp)": "Start", "Stop (bp)": "End",})

# File 2
# 读取 Excel
df_2 = pd.read_excel(WORK_PATH / "2-soybean-quality-region-1.xlsx")

# 提取需要的三列
region_df_2 = df_2[["Chr", "Start (bp)", "Stop (bp)"]].copy()

# 去除 Start == Stop 的行
region_df_2 = region_df_2[region_df_2["Start (bp)"] != region_df_2["Stop (bp)"]].copy()

# Chr：Gm01 -> 1，Gm02 -> 2，Gm15 -> 15
region_df_2["Chr"] = (region_df_2["Chr"].str.replace("Gm", "", regex=False).astype(int))

# 重命名列
region_df_2 = region_df_2.rename(columns={"Start (bp)": "Start", "Stop (bp)": "End",})

# File 3
df_3 = pd.read_excel(WORK_PATH / "3-soybean-quality-region-2.xlsx")

region_df_3 = df_3[["Chr", "Start (bp)", "End (bp)"]].copy()

region_df_3 = region_df_3.dropna(subset=["End (bp)"])

region_df_3 = region_df_3[region_df_3["Start (bp)"] != region_df_3["End (bp)"]].copy()

region_df_3 = region_df_3.rename(columns={"Start (bp)": "Start", "End (bp)": "End",})

region_df_3[["Start", "End"]] = region_df_3[["Start", "End"]].astype(int)

# 合并三个 DataFrame
region_df = pd.concat([region_df_1, region_df_2, region_df_3], ignore_index=True)

# 确保 Start 和 End 是数值类型
region_df["Start"] = pd.to_numeric(region_df["Start"])
region_df["End"] = pd.to_numeric(region_df["End"])

region_df.to_csv(WORK_PATH.parent / "soybean-region.txt",
    sep="\t",
    index=False
)